In [9]:
import pandas as pd

In [10]:
# Path to GLSEA SST data
usace_file = '/Users/ljob/Desktop/Data/USACE/ForecasterPick_merged.csv'
nbs_p_file = '/Users/ljob/Desktop/CNBS_forecast_ver_anom_sst_swe.csv'

#ver_file = 'CNBS_forecast_ver.csv'
#bias_file = 'CNBS_forecast_ver_swe.csv'

existing_model_data = pd.read_csv(usace_file,sep='\t')
my_model_data = pd.read_csv(nbs_p_file,sep='\t')


In [11]:
import pandas as pd
import numpy as np


# ----------------------------
# METRICS
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return np.nan

    return 1 - (ss_res / ss_tot)


def bias(y_true, y_pred):
    # positive = model overpredicts
    # negative = model underpredicts
    return np.mean(y_pred - y_true)


def calc_metrics(g):
    return pd.Series({
        "my_rmse": rmse(g["obs"], g["forecast"]),
        "existing_rmse": rmse(g["obs"], g["existing_forecast"]),

        "my_r2": r2_score(g["obs"], g["forecast"]),
        "existing_r2": r2_score(g["obs"], g["existing_forecast"]),

        "my_bias": bias(g["obs"], g["forecast"]),
        "existing_bias": bias(g["obs"], g["existing_forecast"]),
    })


def add_improvement_columns(df, existing_model_name="existing_model"):
    df = df.copy()

    # RMSE: lower is better
    df["rmse_improvement_pct"] = (
        (df["existing_rmse"] - df["my_rmse"])
        / df["existing_rmse"]
        * 100
    )

    # R2: higher is better
    df["r2_improvement"] = df["my_r2"] - df["existing_r2"]

    # Bias: closer to zero is better
    df["bias_improvement_pct"] = np.where(
        np.abs(df["existing_bias"]) == 0,
        np.nan,
        (
            (np.abs(df["existing_bias"]) - np.abs(df["my_bias"]))
            / np.abs(df["existing_bias"])
            * 100
        )
    )

    df["rmse_winner"] = np.where(
        df["rmse_improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    df["r2_winner"] = np.where(
        df["r2_improvement"] > 0,
        "my_model",
        existing_model_name
    )

    df["bias_winner"] = np.where(
        df["bias_improvement_pct"] > 0,
        "my_model",
        existing_model_name
    )

    return df


# ----------------------------
# STANDARDIZE EXISTING MODEL
# ----------------------------
def standardize_existing_nbs_model(
    existing_df,
    date_col="forecast_date",
    month_col="year_month",
    model_name="existing_model"
):
    df = existing_df.copy()

    df = df.rename(columns={
        date_col: "cfs_run",
        month_col: "forecast_month"
    })

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])
    df["existing_model"] = model_name

    return df


# ----------------------------
# MELT YOUR MODEL DATA
# ----------------------------
def melt_my_nbs_model(my_df):
    df = my_df.copy()

    df["cfs_run"] = pd.to_datetime(df["cfs_run"])
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])

    id_cols = ["cfs_run", "forecast_month", "model"]

    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs") or c.endswith("_nbs_obs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="value"
    )

    long["type"] = np.where(
        long["variable"].str.endswith("_obs"),
        "obs",
        "forecast"
    )

    long["variable"] = long["variable"].str.replace("_obs", "", regex=False)

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    wide = (
        long
        .pivot_table(
            index=["cfs_run", "forecast_month", "model", "lake", "component"],
            columns="type",
            values="value"
        )
        .reset_index()
    )

    return wide


# ----------------------------
# MELT EXISTING MODEL DATA
# ----------------------------
def melt_existing_nbs_model(existing_df):
    df = existing_df.copy()

    id_cols = ["cfs_run", "forecast_month", "existing_model"]

    nbs_cols = [
        c for c in df.columns
        if c.endswith("_nbs")
    ]

    long = df.melt(
        id_vars=id_cols,
        value_vars=nbs_cols,
        var_name="variable",
        value_name="existing_forecast"
    )

    split = long["variable"].str.split("_", n=1, expand=True)
    long["lake"] = split[0]
    long["component"] = split[1]

    return long.drop(columns="variable")


# ----------------------------
# COMPARE YOUR MODEL TO EXISTING NBS MODEL
# ----------------------------
def compare_my_model_to_existing_nbs(
    my_df,
    existing_df,
    existing_model_name="existing_model"
):
    """
    Compares your NBS forecasts to an existing NBS-only model.

    Positive rmse_improvement_pct:
        Your model has lower RMSE.

    Positive r2_improvement:
        Your model has higher R2.

    Positive bias_improvement_pct:
        Your model has bias closer to zero.
    """

    existing_standard = standardize_existing_nbs_model(
        existing_df,
        model_name=existing_model_name
    )

    my_long = melt_my_nbs_model(my_df)
    existing_long = melt_existing_nbs_model(existing_standard)

    merged = my_long.merge(
        existing_long,
        on=["cfs_run", "forecast_month", "lake", "component"],
        how="inner"
    )

    merged = merged.dropna(subset=["forecast", "obs", "existing_forecast"])

    comparison = (
        merged
        .groupby(["model", "lake", "component"])
        .apply(calc_metrics)
        .reset_index()
    )

    comparison = add_improvement_columns(
        comparison,
        existing_model_name=existing_model_name
    )

    comparison = comparison.sort_values(
        "rmse_improvement_pct",
        ascending=False
    )

    overall = (
        merged
        .groupby("model")
        .apply(calc_metrics)
        .reset_index()
    )

    overall = add_improvement_columns(
        overall,
        existing_model_name=existing_model_name
    )

    overall = overall.sort_values(
        "rmse_improvement_pct",
        ascending=False
    )

    return comparison, overall


# ----------------------------
# USAGE
# ----------------------------
comparison, overall = compare_my_model_to_existing_nbs(
    my_df=my_model_data,
    existing_df=existing_model_data,
    existing_model_name="existing_model"
)

print("\n=== NBS Skill by Model/Lake/Component ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(comparison)


=== NBS Skill by Model/Lake/Component ===
   model            lake component     my_rmse  existing_rmse     my_r2  \
10    RF         ontario       nbs  109.532190     112.606889  0.508806   
9     RF  michigan-huron       nbs   45.688204      45.977396  0.607893   
11    RF        superior       nbs   48.280100      48.291734  0.567973   
8     RF            erie       nbs  110.630049     110.257696  0.465596   
3     GP        superior       nbs   53.947888      48.291734  0.460585   
2     GP         ontario       nbs  133.257551     112.606889  0.272968   
15   XGB        superior       nbs   57.523137      48.291734  0.386719   
12   XGB            erie       nbs  132.028967     110.257696  0.238865   
14   XGB         ontario       nbs  136.607092     112.606889  0.235960   
0     GP            erie       nbs  133.930397     110.257696  0.216784   
13   XGB  michigan-huron       nbs   58.453170      45.977396  0.358182   
1     GP  michigan-huron       nbs   60.858647      45.97

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_7633/1356258932.py:224: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)
/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_7633/1356258932.py:241: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_metrics)


In [12]:
print("\n=== Overall NBS Skill by Model ===")
print(overall)


=== Overall NBS Skill by Model ===
  model     my_rmse  existing_rmse     my_r2  existing_r2    my_bias  \
2    RF   84.638479      85.561503  0.582917     0.573771  -1.152499   
0    GP  102.845850      85.561503  0.384171     0.573771  -3.209896   
3   XGB  103.463556      85.561503  0.376751     0.573771   0.887035   
1    NN  124.633984      85.561503  0.095602     0.573771 -16.372726   

   existing_bias  rmse_improvement_pct  r2_improvement  bias_improvement_pct  \
2     -16.596522              1.078785        0.009147             93.055782   
0     -16.596522            -20.201079       -0.189600             80.659224   
3     -16.596522            -20.923023       -0.197019             94.655293   
1     -16.596522            -45.665958       -0.478168              1.348450   

      rmse_winner       r2_winner bias_winner  
2        my_model        my_model    my_model  
0  existing_model  existing_model    my_model  
3  existing_model  existing_model    my_model  
1  existin